# Employee Performance and Attrition Analytics Using Data Analytics
**B.Tech Computer Science Engineering Academic Project**

### Objectives:
1. Analyze employee performance distribution across roles and departments.
2. Uncover the root causes behind employee attrition.
3. Perform thorough data cleaning, encoding, and exploratory data analysis.
4. Train and benchmark supervised Machine Learning algorithms (Logistic Regression, Decision Tree, Random Forest).
5. Provide actionable decision support recommendations for Human Resources leadership.

In [ ]:
# Step 1: Import Core Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling
sns.set_theme(style='whitegrid')
%matplotlib inline

print('Libraries successfully imported.')

In [ ]:
# Step 2: Load the Employee Dataset
dataset_path = '../dataset/employee_data.csv'
df = pd.read_csv(dataset_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# Step 3: Dataset Health & Sanity Check
print('--- Dataset Info ---')
df.info()
print('\n--- Missing Values Check ---')
print(df.isnull().sum())
print(f'\nDuplicate Records: {df.duplicated(subset=["EmployeeID"]).sum()}')

In [ ]:
# Step 4: Exploratory Data Analysis - Demographics & Attrition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Attrition Count
sns.countplot(data=df, x='Attrition', palette=['#27ae60', '#c0392b'], ax=axes[0])
axes[0].set_title('Overall Attrition Count (Stayed vs Left)', weight='bold')

# Department Breakdown
sns.countplot(data=df, x='Department', hue='Attrition', palette=['#27ae60', '#c0392b'], ax=axes[1])
axes[1].set_title('Attrition Breakdown by Department', weight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Step 5: OverTime Impact on Attrition
plt.figure(figsize=(7, 4.5))
ot_crosstab = pd.crosstab(df['OverTime'], df['Attrition'], normalize='index') * 100
ot_crosstab.plot(kind='bar', stacked=True, color=['#27ae60', '#c0392b'], figsize=(7, 4.5))
plt.title('Impact of Mandatory OverTime on Attrition Rate (%)', weight='bold')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.show()
print(ot_crosstab)

In [ ]:
# Step 6: Employee Performance Analytics
# Compute Normalized Composite Performance Score (0 - 100)
rating_part = (df['PerformanceRating'] / 4.0) * 50.0
training_part = (df['TrainingTimesLastYear'].clip(upper=6) / 6.0) * 20.0
attendance_part = (1.0 - (df['Absenteeism'].clip(upper=25) / 25.0)) * 15.0
satisfaction_part = (df['JobSatisfaction'] / 4.0) * 15.0

df['CompositeScore'] = (rating_part + training_part + attendance_part + satisfaction_part).round(2)

plt.figure(figsize=(8, 4.5))
sns.histplot(df['CompositeScore'], kde=True, color='#2980b9')
plt.title('Workforce Composite Performance Score Distribution (0-100)', weight='bold')
plt.xlabel('Composite Performance Score')
plt.show()

In [ ]:
# Step 7: Correlation Heatmap
num_features = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Employee Metrics', weight='bold')
plt.show()

In [ ]:
# Step 8: Preprocessing & Feature Engineering for Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Target & Binary Encoding
y = df['Attrition'].map({'Yes': 1, 'No': 0})
df_ml = df.drop(columns=['EmployeeID', 'Attrition', 'CompositeScore'])
df_ml['OverTime'] = df_ml['OverTime'].map({'Yes': 1, 'No': 0})

# One-Hot Encoding for categorical features
cat_cols = ['Department', 'JobRole', 'Education', 'Gender']
X = pd.get_dummies(df_ml, columns=cat_cols, drop_first=True, dtype=int)

# Stratified 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standard Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Training Samples: {X_train.shape[0]}, Testing Samples: {X_test.shape[0]}')
print(f'Total Features: {X.shape[1]}')

In [ ]:
# Step 9: Model Training & Evaluation
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
}

benchmark = []
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    
    benchmark.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(benchmark)
results_df

In [ ]:
# Step 10: Top Predictive Features
rf = models['Random Forest']
fi = pd.DataFrame({'Feature': X.columns, 'Importance': rf.feature_importances_})
fi = fi.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=fi, x='Importance', y='Feature', palette='viridis')
plt.title('Top 10 Feature Importances for Attrition Risk', weight='bold')
plt.xlabel('Importance Score')
plt.show()

### Conclusion & HR Strategic Summary
1. **Primary Attrition Drivers:** Mandatory Overtime, Low Monthly Compensation, and Low Job Satisfaction.
2. **Retention Recommendations:** Balance workload across project cycles, conduct periodic salary benchmarking, and introduce structured career growth plans to eliminate promotion stagnation.
3. **Dashboard:** Launch the Streamlit application for interactive executive presentation via `streamlit run dashboard/app.py`.